## Project: Building Your First Vector Search System

# Mission
Build a functional vector search system that demonstrates the core concepts: collections, points, similarity search, and filtering. You’ll design simple 4-dimensional vectors that represent different concepts or items.

# What to Build
A working search system with:

1. One collection with 4-dimensional vectors and Cosine distance
2. 5–10 points with hand-crafted vectors and meaningful payloads
2. Basic similarity search to find nearest neighbors
4. Filtered search combining similarity with payload conditions

# Setup
a. Prerequisites
1. Qdrant Cloud cluster (URL + API key)
2. Python 3.9+ (or Colab)
3. Required packages: qdrant-client.

b. Models - None. We will create vectors by hand.

c. Dataset - None. We will create our own data points.

>Before creating data, decide what each of the four dimensions in your vectors will represent. This is the creative part of vector search!

Example Ideas:

1. Product categories: Create vectors where each dimension represents a feature (affordability, quality, popularity, innovation). Electronics might be [0.8, 0.7, 0.9, 0.6], while books could be [0.3, 0.9, 0.4, 0.8].
2. Color palettes: Each dimension represents color (red, green, blue). Bright red: [0.9, 0.1, 0.1], forest green: [0.1, 0.8, 0.2].
3. Data types: Dimensions for structure, size, complexity, frequency. Spreadsheets: [0.9, 0.6, 0.3, 0.7], images: [0.2, 0.8, 0.5, 0.4].
4. Movie genres: Action, drama, comedy, sci-fi intensities. Action thriller: [0.9, 0.3, 0.1, 0.7], romantic comedy: [0.1, 0.6, 0.9, 0.2].
> For this tutorial, we’ll use the Product Categories concept.

# Step 1: Initialize Client

In [1]:
from qdrant_client import QdrantClient, models
import os
from dotenv import load_dotenv

load_dotenv()

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

# Step 2: Create Collection

In [2]:
collection_name = "day0_first_system"
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(size=4, distance=models.Distance.COSINE),
)

# Create payload index right after creating the collection and before uploading any data to enable filtering.
# If you add it later, HNSW won't rebuild automatically—bump ef_construct (e.g., 100→101) to trigger a safe rebuild.
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

# Step 3: Insert Points

In [5]:
points=[
    models.PointStruct(
        id=1,
        vector=[0.9, 0.1, 0.1, 0.8], # High affordability, high innovation
        payload={"name": "Budget Smartphone", "category": "electronics", "price": 299},
    ),
    models.PointStruct(
        id=2,
        vector=[0.2, 0.9, 0.8, 0.5], # High quality, high popularity
        payload={"name": "Bestselling Novel", "category": "books", "price": 19},
    ),
    models.PointStruct(
        id=3,
        vector=[0.8, 0.3, 0.2, 0.9], # High affordability, high innovation (similar to ID 1)
        payload={"name": "Smart Home Hub", "category": "electronics", "price": 89},
    ),
    models.PointStruct(
        id=4,
        vector=[0.1, 0.8, 0.9, 0.4], # High quality, high popularity (similar to ID 2)
        payload={"name": "Classic Literature Set", "category": "books", "price": 49},
    ),
    models.PointStruct(
        id=5,
        vector=[0.7, 0.4, 0.3, 0.8], # High affordability, moderate innovation
        payload={"name": "Eco-Friendly Water Bottle", "category": "sports", "price": 25},
    )
]

client.upsert(collection_name=collection_name, points=points)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

# Step 4: Test Searches

In [7]:
# Define a query vector for "affordable and innovative"
query_vector = [0.85, 0.2, 0.1, 0.9]

# 1. Basic similarity search
basic_results = client.query_points(collection_name, query=query_vector)

# 2. Filtered search (only find electronics)
filtered_results = client.query_points(
    collection_name,
    query=query_vector,
    query_filter=models.Filter(
        must=[models.FieldCondition(key="category", match=models.MatchValue(value="sports"))]
    ),
)
print("Filtered search results:", filtered_results)

Filtered search results: points=[ScoredPoint(id=5, version=4, score=0.96428066, payload={'name': 'Eco-Friendly Water Bottle', 'category': 'sports', 'price': 25}, vector=None, shard_key=None, order_value=None)]
